In [ ]:
import pandas as pd
from utils import classify_customer


customers = pd.read_csv("data/customers.csv")
orders    = pd.read_csv("data/orders.csv")
products  = pd.read_csv("data/products.csv")

for name, df in [("customers", customers), ("orders", orders), ("products", products)]:
    print(f"\n=== {name} ===")
    print("Shape:", df.shape)
    df.info()
    print(df.describe())
    print("Nulls:\n", df.isnull().sum())

# ── Task 2.2: Clean customers.csv ────────────────────────────────────

# Fill nulls
customers["age"].fillna(customers["age"].median(), inplace=True)
customers["gender"].fillna("Unknown", inplace=True)

# Remove duplicate emails (keep first)
customers.drop_duplicates(subset=["email"], keep="first", inplace=True)

# Convert gender to category dtype
customers["gender"] = customers["gender"].astype("category")

# Add age_group using classify_customer from utils.py
customers["age_group"] = customers["age"].apply(classify_customer)

# Save cleaned result
customers.to_csv("outputs/cleaned_customers.csv", index=False)
print("Saved cleaned_customers.csv")
print(customers.head())

# ── Task 2.3: Clean products.csv ─────────────────────────────────────

# Fix price column — stored as string like "₹1200" or "1200.0"
products["price"] = (
    products["price"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
products["price"] = pd.to_numeric(products["price"], errors="coerce")

# Fill null rating with mean (rounded to 1 decimal)
mean_rating = round(products["rating"].mean(), 1)
products["rating"].fillna(mean_rating, inplace=True)

# Convert category to category dtype
products["category"] = products["category"].astype("category")

print(products.dtypes)
print(products.head())

# ── Task 2.4: Null Report Function ───────────────────────────────────

def null_report(df, name):
    """Prints a clean null summary for a DataFrame."""
    print(f"\n=== Null Report: {name} ===")
    total = len(df)
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print("  No nulls found!")
    else:
        for col, count in nulls.items():
            pct = count / total * 100
            print(f"  {col:<20} -> {count} nulls ({pct:.1f}%)")

# Run on all three DataFrames BEFORE cleaning (reload to show original)
customers_raw = pd.read_csv("data/customers.csv")
orders_raw    = pd.read_csv("data/orders.csv")
products_raw  = pd.read_csv("data/products.csv")

print("BEFORE CLEANING:")
null_report(customers_raw, "customers")
null_report(orders_raw,    "orders")
null_report(products_raw,  "products")

print("\nAFTER CLEANING:")
null_report(customers, "customers")
null_report(orders,    "orders")
null_report(products,  "products")


In [ ]:
import numpy as np

# Task 3.1 — Revenue Array

price_array =products['price'].to_numpy()



price_with_tax = price_array * 1.18

final_price = price_with_tax * 0.90

print("shape",final_price.shape)
print("dtype",type(price_array))


print("first 5 values of final_price",final_price[:5])


# Task 3.2 — Reshape Exercise


print(" first 20 product prices (after tax + discount)",final_price[:20])

print(" reshape(4,5)",final_price[:20].reshape(4,5))

print(" Transpose",final_price[:20].reshape(4,5).T)

print(" flatten",final_price[:20].reshape(4,5).T.flatten())

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("outputs/retailsense.db")

customers = pd.read_csv("outputs/cleaned_customers.csv")
products_clean =  products.copy()
orders = pd.read_csv("data/orders.csv")

# Write your cleaned DataFrames to the database
customers.to_sql("customers", conn, if_exists="replace", index=False)
products_clean.to_sql("products", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)

# Task 4.2 — Answer These Business Questions Using SQL
def sql(q):
    return pd.read_sql_query(q, conn)

# Q1: Top 5 cities by number of orders
print("\nQ1: Top 5 cities by orders")
print(sql("""
    SELECT 
city,
count(customer_id) as customer_count 
FROM customers
GROUP BY city
ORDER BY customer_count DESC
"""))
# Q2	What are the top 5 most expensive products? Show product name and price.
print("\nQ2: top 5 most expensive products")
print(sql("""
    SELECT 
product_name, price
FROM products
ORDER BY price DESC
LIMIT 5
"""))
# Q3	Which orders had a discount greater than 20%? Show order_id, customer_id, discount_pct.

print("\nQ3: orders had a discount greater than 20%")
print(sql("""
    SELECT 
order_id, customer_id, discount_pct
FROM orders
WHERE discount_pct > 20
"""))

# Q4	What is the average rating per product category?
print("\nQ4: the average rating per product category")
print(sql("""
    SELECT category,
           ROUND(AVG(rating), 2) AS avg_rating
    FROM products
    GROUP BY category
    ORDER BY avg_rating DESC
"""))

print("\nQ5: Categories with >10 products AND avg rating > 3.5")
print(sql("""
    SELECT category,
           COUNT(product_id) AS product_count,
           ROUND(AVG(rating), 2) AS avg_rating
    FROM products
    GROUP BY category
    HAVING COUNT(product_id) > 10 AND AVG(rating) > 3.5
"""))



# Q6: Each order — customer name, product name, quantity, base price
print("\nQ6: Order details (3-table JOIN)")
print(sql("""
    SELECT c.name AS customer_name,
           p.product_name,
           o.quantity,
           p.price AS base_price
    FROM orders o
    INNER JOIN customers c ON o.customer_id = c.customer_id
    INNER JOIN products  p ON o.product_id  = p.product_id
"""))

# Q7: Customers with NO orders
print("\nQ7: Customers with no orders")
print(sql("""
    SELECT c.name, c.email
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.order_id IS NULL
"""))

conn.close()
print("\nAll SQL queries complete!")

In [ ]:
from utils import calculate_revenue

# ── Task 5.1: Merge DataFrames ───────────────────────────────────────
customers = pd.read_csv("outputs/cleaned_customers.csv")
products_clean =  products.copy()
orders = pd.read_csv("data/orders.csv")

# Merge orders + customers (inner join on customer_id)
merged = orders.merge(customers, on="customer_id", how="inner")

# Merge result + products (inner join on product_id)
full_df = merged.merge(products_clean, on="product_id", how="inner")

# Add revenue column using calculate_revenue from utils.py
full_df["revenue"] = full_df.apply(
    lambda row: calculate_revenue(row["price"], row["quantity"], row["discount_pct"]),
    axis=1
)

print("full_df shape:", full_df.shape)
print(full_df[["name","product_name","quantity","price","discount_pct","revenue"]].head())

# ── Task 5.2: GroupBy Analysis ───────────────────────────────────────

# 1. Total revenue per product category
print("\n1. Total revenue per category:")
rev_by_cat = full_df.groupby("category")["revenue"].sum().sort_values(ascending=False)
print(rev_by_cat)

# 2. Average order quantity per city
print("\n2. Avg order quantity per city:")
avg_qty_city = full_df.groupby("city")["quantity"].mean().sort_values(ascending=False)
print(avg_qty_city)

# 3. Top 3 customers by total revenue
print("\n3. Top 3 customers by revenue:")
top3 = full_df.groupby("name")["revenue"].sum().nlargest(3)
print(top3)

# 4. Custom aggregation per category
print("\n4. Custom aggregation per category:")
custom_agg = full_df.groupby("category").agg(
    total_revenue    = ("revenue",     "sum"),
    unique_customers = ("customer_id", "nunique"),
    avg_discount     = ("discount_pct","mean")
).round(2)
print(custom_agg)

# ── Task 5.3: Pivot Table ────────────────────────────────────────────

pivot = full_df.pivot_table(
    index="category",
    columns="age_group",
    values="revenue",
    aggfunc="sum"
).fillna(0).round(2)

print("\nPivot Table — Revenue by Category vs Age Group:")
print(pivot)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import requests
import json

# ── Part 6: Data Visualization ───────────────────────────────────────

customers = pd.read_csv("outputs/cleaned_customers.csv")
products_clean =  products.copy()
orders = pd.read_csv("data/orders.csv")

# P1: Revenue by category — Matplotlib bar chart
rev_cat = full_df.groupby("category")["revenue"].sum().sort_values(ascending=False)

plt.figure(figsize=(7, 4))
plt.bar(rev_cat.index, rev_cat.values, color="steelblue")
plt.title("Revenue by Product Category")
plt.xlabel("Category")
plt.ylabel("Total Revenue (₹)")
plt.tight_layout()
plt.show()
# Insight: Electronics drives the highest revenue among all categories.

# P2: Customer age distribution — Seaborn histogram
plt.figure(figsize=(7, 4))
sns.histplot(customers["age"], bins=10, kde=True, color="coral")
plt.title("Customer Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.tight_layout()
plt.show()
# Insight: Most customers are between 25–45, the core Adult segment.

# P3: Price distribution per category — Seaborn box plot
plt.figure(figsize=(7, 4))
sns.boxplot(data=products_clean, x="category", y="price", palette="Set2")
plt.title("Price Distribution per Category")
plt.xlabel("Category")
plt.ylabel("Price (₹)")
plt.tight_layout()
plt.show()
# Insight: Electronics has the widest price range and highest median price.

# P4: Correlation heatmap of numeric columns in full_df
num_cols = full_df.select_dtypes(include="number")
plt.figure(figsize=(8, 5))
sns.heatmap(num_cols.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap — full_df Numeric Columns")
plt.tight_layout()
plt.show()
# Insight: Quantity and revenue are positively correlated; discount reduces revenue.

# P5: Interactive scatter — Price vs Rating colored by category
fig = px.scatter(
    products_clean,
    x="price", y="rating",
    color="category",
    hover_name="product_name",
    title="Price vs Rating by Category",
    labels={"price": "Price (₹)", "rating": "Rating"}
)
fig.show()
# Insight: Higher-priced products don't always have better ratings.

# P6 (Bonus): Top 10 customers by revenue — interactive bar
top10 = full_df.groupby("name")["revenue"].sum().nlargest(10).reset_index()
fig2 = px.bar(top10, x="name", y="revenue",
              title="Top 10 Customers by Revenue",
              labels={"name": "Customer", "revenue": "Revenue (₹)"},
              color="revenue")
fig2.show()

# ── Part 7: API Integration ───────────────────────────────────────────

# Task 7.1 — GET request
response = requests.get("https://jsonplaceholder.typicode.com/users")
print(f"Status code: {response.status_code}")

users = response.json()
print(f"\nFetched {len(users)} users:")
for user in users:
    print(f"  Name: {user['name']}, Email: {user['email']}, City: {user['address']['city']}")

# Task 7.2 — POST request
post_body = {
    "title":  "RetailSense Report",
    "body":   "Q4 revenue exceeded targets by 12%",
    "userId": 1
}
post_response = requests.post(
    "https://jsonplaceholder.typicode.com/posts",
    json=post_body
)
print(f"\nPOST status code: {post_response.status_code}")  # expect 201
post_data = post_response.json()
print(f"Created post id: {post_data['id']}")

# Task 7.3 — Save API data to file and reload
with open("outputs/api_users.json", "w") as f:
    json.dump(users, f, indent=2)
print("\nSaved api_users.json")

with open("outputs/api_users.json", "r") as f:
    reloaded = json.load(f)
print(f"First user name (reloaded): {reloaded[0]['name']}")

## Part 9 — Ethics & Privacy Audit

### 1. PII Identification
| Column       | Dataset   | PII? | Action |
|-------------|-----------|------|--------|
| name        | customers | Yes  | Mask or pseudonymize before sharing |
| email       | customers | Yes  | Drop entirely — not needed for ML |
| customer_id | all       | Yes  | Hash (SHA-256) to anonymize linkage |
| city        | customers | Low  | Keep — aggregate-level, not precise |

### 2. Anonymization Plan
Before sharing with a third-party ML vendor:
- DROP: email (direct identifier, not needed)
- HASH: customer_id using SHA-256 so records can't be re-linked
- MASK: name → replace with "Customer_001" style IDs
- KEEP: city, age_group, category (aggregate, not personal)
- GENERALIZE: age → age_group buckets already done

### 3. API Ethics — Three things to check in ToS
1. **Data storage policy** — does the API store our requests/data on their servers?
2. **Rate limits & commercial use** — are we allowed to use it in a production product?
3. **Data ownership** — who owns the data returned by the API?

### 4. Null Handling Bias
Filling null ages with the median pulls all unknown-age customers
toward the middle age range (Adult segment). This means Youth and
Senior segments may be underrepresented in age_group analysis,
since customers with missing ages are artificially classified as
Adults. A better approach would be to keep nulls as "Unknown"
and analyze them as a separate group.